# Import libs

In [1]:
import numpy as np
from tqdm.auto import tqdm
import collections
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import TrainingArguments
from transformers import Trainer
import evaluate

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [2]:
from huggingface_hub import notebook_login
notebook_login()

# Configure

In [3]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 384
STRIDE = 128

# Setup Datasets

In [4]:
DATASET_NAME = 'squad_v2'
raw_datasets = load_dataset(DATASET_NAME)

In [5]:
raw_datasets['train'][1]

{'id': '56be85543aeaaa14008c9065',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': 'What areas did Beyonce compete in when she was growing up?',
 'answers': {'text': ['singing and dancing'], 'answer_start': [207]}}

In [25]:
raw_datasets['validation'][22]['answers']

{'text': ['King Charles III', 'King Charles III', 'King Charles III'],
 'answer_start': [324, 324, 324]}

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

c:\Users\Admin\miniconda3\envs\tf-gpu\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# Preprocessing data

In [7]:
def preprocess_training_examples(examples):
    # Get questions from examples
    # and remove redundant spaces
    questions = [q.strip() for q in examples["question"]]

    # tokenize input data
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        stride=STRIDE,
        padding="max_length",
    )
    # Extract offset mappings from inputs
    # then pop it from inputs
    offset_mapping = inputs.pop("offset_mapping")

    # Extract sample mappings from inputs
    # then pop it from inputs
    sample_map = inputs.pop("overflow_to_sample_mapping")

    # get answers from examples
    answers = examples["answers"]

    # Initiate start end stop answer position list
    start_positions = []
    end_positions = []

    # Loop through offset_mapping
    for i, offset in enumerate(offset_mapping):
        # identify index of sample relate to the current offset
        sample_idx = sample_map[i]
        # get sequence_ids from input
        sequence_ids = inputs.sequence_ids(i)
        # Get start and end position of context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # get answer for this sample
        answer = answers[sample_idx]
        if len(answer["text"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
        else:
            start_char = answer["answer_start"][0]
            end_char = answer["answer_start"][0] + len(answer["text"][0])
            # if the answer is not in the context
            if (
                offset[context_start][0] > start_char
                or offset[context_end][1] < end_char
            ):
                start_positions.append(0)
                end_positions.append(0)
            else:
                # else set the start and end position
                idx = context_start
                while idx <= context_end and offset[idx][0] <= start_char:
                    idx += 1
                start_positions.append(idx - 1)
                idx = context_end
                while idx >= context_start and offset[idx][1] >= end_char:
                    idx -= 1
                end_positions.append(idx + 1)

    # adding start, end position to inputs
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [8]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

In [64]:
len(raw_datasets["train"]), len(train_dataset)

(130319, 131754)

In [9]:
def preprocess_validation_examples(examples):
    # Get questions from examples
    # and remove redundant spaces
    questions = [q.strip() for q in examples["question"]]

    # tokenize input data
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        stride=STRIDE,
        padding="max_length",
    )

    # Extract sample mappings from inputs
    # then pop it from inputs
    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    # Xác định ví dụ tham chi ếu cho mỗi dòng đầu vào và
    # điều chỉnh ánh xạ offset
    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        # Loại bỏ các offset không phù hợp với sequence_ids
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]
    # Thêm thông tin ví dụ tham chi ếu vào đầu vào
    inputs["example_id"] = example_ids
    return inputs

In [10]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
# In ra độ dài của raw_datasets [" validation "]
# và validation_dataset để so sánh.
len(raw_datasets["validation"]), len(validation_dataset)

(11873, 12134)

# Training

In [11]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
args = TrainingArguments(
    output_dir="distilbert-finetuned-squadv2",  
    evaluation_strategy="no",  # Chế độ đánh giá không tự động sau mỗi epoch
    save_strategy="epoch",  # Lưu checkpoint sau mỗi epoch
    learning_rate=2e-5,  # Tốc độ học
    num_train_epochs=3,  # Số epoch huấn luy ện
    weight_decay=0.01,  # Giảm trọng lượng mô hình để tránh overfitting
    fp16=True,  # Sử dụng kiểu dữ liệu half - precision để tối ưu tài nguyên
    push_to_hub=True,  # Đẩy kết quả huấn luyện lên HuggingFace Hub
    per_device_train_batch_size=32
)

c:\Users\Admin\miniconda3\envs\tf-gpu\lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Khởi tạo một đối tượng Trainer để huấn luy ện mô hình
trainer = Trainer(
    model=model,  # Sử dụng mô hình đã tạo trước đó
    args=args,  # Các tham số và cấu hình huấn luy ện
    train_dataset=train_dataset,  # Sử dụng tập dữ liệu huấn luyện
    eval_dataset=validation_dataset,  # Sử dụng tập dữ liệu đánh giá
    tokenizer=tokenizer,  # Sử dụng tokenizer để xử lý văn bản
)
# Bắt đầu quá trình huấn luy ện
trainer.train()

c:\Users\Admin\miniconda3\envs\tf-gpu\lib\site-packages\accelerate\accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


  0%|          | 0/12354 [00:00<?, ?it/s]

{'loss': 1.233, 'grad_norm': 7.360753059387207, 'learning_rate': 1.9193783389995144e-05, 'epoch': 0.12}
{'loss': 1.0245, 'grad_norm': 10.968137741088867, 'learning_rate': 1.8384328962279426e-05, 'epoch': 0.24}
{'loss': 1.146, 'grad_norm': 16.502262115478516, 'learning_rate': 1.7574874534563705e-05, 'epoch': 0.36}
{'loss': 1.3165, 'grad_norm': 12.957315444946289, 'learning_rate': 1.6765420106847984e-05, 'epoch': 0.49}
{'loss': 1.2656, 'grad_norm': 9.95414924621582, 'learning_rate': 1.5957584587987698e-05, 'epoch': 0.61}
{'loss': 1.222, 'grad_norm': 11.264152526855469, 'learning_rate': 1.5148130160271978e-05, 'epoch': 0.73}
{'loss': 1.2161, 'grad_norm': 11.579215049743652, 'learning_rate': 1.4338675732556257e-05, 'epoch': 0.85}
{'loss': 1.1888, 'grad_norm': 11.684990882873535, 'learning_rate': 1.3529221304840539e-05, 'epoch': 0.97}
{'loss': 0.9958, 'grad_norm': 10.476509094238281, 'learning_rate': 1.271976687712482e-05, 'epoch': 1.09}
{'loss': 0.9552, 'grad_norm': 12.53016471862793, 'lea

TrainOutput(global_step=12354, training_loss=0.9675579345861222, metrics={'train_runtime': 1802.9911, 'train_samples_per_second': 219.226, 'train_steps_per_second': 6.852, 'total_flos': 3.873165421863629e+16, 'train_loss': 0.9675579345861222, 'epoch': 3.0})

In [ ]:
trainer.push_to_hub(commit_message="Reader_Squadv2")

events.out.tfevents.1739883427.DESKTOP-502NHKM.15004.1:   0%|          | 0.00/10.2k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/binhphap5/distilbert-finetuned-squadv2/commit/f08a2f94d0452f9072b671babe8f3b2fa9eaa474', commit_message='Reader_Squadv2', commit_description='', oid='f08a2f94d0452f9072b671babe8f3b2fa9eaa474', pr_url=None, repo_url=RepoUrl('https://huggingface.co/binhphap5/distilbert-finetuned-squadv2', endpoint='https://huggingface.co', repo_type='model', repo_id='binhphap5/distilbert-finetuned-squadv2'), pr_revision=None, pr_num=None)

# Evaluation

In [19]:
metric = evaluate.load("squad_v2")


In [22]:
N_BEST = 20  # Số lượng kết quả tốt nhất được lựa chọn sau khi dự đoán
MAX_ANS_LENGTH = 30  # Độ dài tối đa cho câu trả lời dự đoán


def compute_metrics(start_logits, end_logits, features, examples):
    # Tạo một từ điển mặc định để ánh xạ mỗi ví dụ
    # với danh sách các đặc trưng tương ứng
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)
        predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []
        # Lặp qua tất cả các đặc trưng liên quan đến ví dụ đó
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]
            # Lấy các chỉ số có giá trị lớn nhất cho start và end logits
            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Bỏ qua các câu trả lời
                    # không hoàn toàn nằm trong ngữ cảnh
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Bỏ qua các câu trả lời có độ dài > max_answer_length
                    if end_index - start_index + 1 > MAX_ANS_LENGTH:
                        continue
                    # Tạo một câu trả lời mới
                    text = context[offsets[start_index][0] : offsets[end_index][1]]
                    logit_score = start_logit[start_index] + end_logit[end_index]
                    answer = {
                        "text": text,
                        "logit_score": logit_score,
                    }
                    answers.append(answer)
        # Chọn câu trả lời có điểm số tốt nhất
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            answer_dict = {
                "id": example_id,
                "prediction_text": best_answer["text"],
                "no_answer_probability": 1 - best_answer["logit_score"],
            }
        else:
            answer_dict = {
                "id": example_id,
                "prediction_text": "",
                "no_answer_probability": 1.0,
            }
        predicted_answers.append(answer_dict)
    # Tạo danh sách câu trả lời lý thuyết từ các ví dụ
    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    # Sử dụng metric.compute để tính toán các độ đo và trả về kết quả
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [23]:
# Thực hiện dự đoán trên tập dữ liệu validation
predictions, _, _ = trainer.predict(validation_dataset)
# Lấy ra thông tin về các điểm bắt đầu và
# điểm kết thúc của câu trả lời dự đoán
start_logits, end_logits = predictions
# Tính toán các chỉ số đánh giá sử dụng hàm compute_metrics
results = compute_metrics(
    start_logits, end_logits, validation_dataset, raw_datasets["validation"]
)
results

  0%|          | 0/1517 [00:00<?, ?it/s]

  0%|          | 0/11873 [00:00<?, ?it/s]

{'exact': 45.59925882253853,
 'f1': 49.59079475784273,
 'total': 11873,
 'HasAns_exact': 74.7132253711201,
 'HasAns_f1': 82.70774395409344,
 'HasAns_total': 5928,
 'NoAns_exact': 16.568544995794785,
 'NoAns_f1': 16.568544995794785,
 'NoAns_total': 5945,
 'best_exact': 63.03377410932367,
 'best_exact_thresh': -11.00390625,
 'best_f1': 64.51376446030127,
 'best_f1_thresh': -10.32421875}

In [26]:
from transformers import pipeline

PIPELINE_NAME = "question-answering"
MODEL_NAME = "binhphap5/distilbert-finetuned-squadv2"
pipe = pipeline(PIPELINE_NAME, model=MODEL_NAME, device=device)

In [32]:
INPUT_QUESTION = "How can I learn NLP ?"
INPUT_CONTEXT = "hi facebook users, today we will learn NLP"
pipe(question=INPUT_QUESTION, context=INPUT_CONTEXT)

{'score': 0.28102192282676697, 'start': 3, 'end': 11, 'answer': 'facebook'}